# Phase 4 B0 setup run

This notebook runs the ten-language Phase 4 B0 sequential baseline only. It does not run M3 or any prior-weighted replay method. Use the private prepared-data ZIP and source package as Kaggle inputs.

Before running:
1. Enable Kaggle GPU accelerator.
2. Add the Kaggle secret `HF_TOKEN` and accept the gated Llama model terms.
3. Attach one private Phase 4 prepared-data dataset and one Phase 4 source dataset.
4. Keep `ALLOW_FRESH_START = False` when attaching an existing progress ZIP.

Run this notebook once per Kaggle session: it selects and runs only the first order without a validated result. After each order, download `phase4_b0_progress.zip` and attach it to the next session to continue. The five orders remain separate experiments; this avoids requiring one Kaggle session to run all five. It does not reduce total GPU time. Results are validated against the frozen Phase 4 protocol.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

SOURCE_ROOT = WORK_ROOT / 'phase4_confirmatory_source'
source_archives = list(INPUT_ROOT.rglob('phase4_confirmatory_source.zip'))
source_roots = [
    requirements.parent.parent
    for requirements in INPUT_ROOT.rglob('requirements-phase2.txt')
    if requirements.parent.name == 'phase2'
    and (requirements.parent / 'run_baseline.py').is_file()
    and (requirements.parent.parent / 'phase4' / 'configs' / 'controlled-xnli-10lang-v1.json').is_file()
]
if len(source_archives) + len(source_roots) != 1:
    visible = sorted(str(path.relative_to(INPUT_ROOT)) for path in INPUT_ROOT.rglob('*') if path.is_file())
    raise FileNotFoundError(
        'Attach exactly one Phase 4 source dataset. Kaggle may expose '
        'phase4_confirmatory_source.zip or its extracted files. '
        f'Found {len(source_archives)} ZIP(s), {len(source_roots)} extracted source root(s). '
        f'Files under /kaggle/input: {visible[:40]}'
    )
if source_roots:
    SOURCE_ROOT = source_roots[0]
else:
    SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source_archives[0]) as archive:
        for member in archive.infolist():
            parts = Path(member.filename).parts
            if '..' in parts or Path(member.filename).is_absolute():
                raise ValueError('The source archive contains an unsafe path.')
        archive.extractall(SOURCE_ROOT)
    if not (SOURCE_ROOT / 'phase2' / 'run_baseline.py').is_file():
        raise FileNotFoundError('The source archive is missing phase2/run_baseline.py.')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(SOURCE_ROOT / 'phase2' / 'requirements-phase2.txt')
], check=True)
sys.path.insert(0, str(SOURCE_ROOT))

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
if not hf_token:
    raise RuntimeError('HF_TOKEN is empty. Add it as a Kaggle secret.')
os.environ['HF_TOKEN'] = hf_token
del hf_token

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU and restart this notebook.')
print('GPU:', torch.cuda.get_device_name(0))
print('Source package:', SOURCE_ROOT)

In [ ]:
import json
import shutil

PROTOCOL_PATH = SOURCE_ROOT / 'phase4' / 'configs' / 'controlled-xnli-10lang-v1.json'
ARTIFACT_ROOT = WORK_ROOT / 'phase4_b0_artifacts_retry'

progress_archives = list(INPUT_ROOT.rglob('phase4_b0_progress.zip'))
progress_roots = [
    manifest.parent
    for manifest in INPUT_ROOT.rglob('protocol_manifest.json')
    if manifest.parent.name == 'phase4_b0_artifacts'
    and (manifest.parent / 'data').is_dir()
    and (manifest.parent / 'results').is_dir()
]
if len(progress_archives) + len(progress_roots) > 1:
    raise ValueError('Attach only one Phase 4 B0 progress dataset.')

if progress_archives:
    with zipfile.ZipFile(progress_archives[0]) as archive:
        archive.extractall(WORK_ROOT)
    extracted_root = WORK_ROOT / 'phase4_b0_artifacts'
    if extracted_root.is_dir() and extracted_root != ARTIFACT_ROOT:
        if ARTIFACT_ROOT.exists() and any(ARTIFACT_ROOT.iterdir()):
            raise FileExistsError(f'Artifact root is not empty: {ARTIFACT_ROOT}')
        if ARTIFACT_ROOT.exists():
            ARTIFACT_ROOT.rmdir()
        extracted_root.replace(ARTIFACT_ROOT)
elif progress_roots:
    shutil.copytree(progress_roots[0], ARTIFACT_ROOT, dirs_exist_ok=True)
else:
    prepared_archives = list(INPUT_ROOT.rglob('phase4_prepared_xnli.zip'))
    prepared_roots = []
    for manifest_path in INPUT_ROOT.rglob('protocol_manifest.json'):
        if not (manifest_path.parent / 'data').is_dir():
            continue
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest.get('protocol_id') == 'phase4-xnli-controlled-10lang-v1':
            prepared_roots.append(manifest_path.parent)
    if len(prepared_archives) + len(prepared_roots) != 1:
        visible = sorted(str(path.relative_to(INPUT_ROOT)) for path in INPUT_ROOT.rglob('*') if path.is_file())
        raise FileNotFoundError(
            'Attach exactly one Phase 4 prepared-data dataset. Kaggle may expose '
            'phase4_prepared_xnli.zip or the extracted protocol_manifest.json and data/ folder. '
            f'Found {len(prepared_archives)} ZIP(s), {len(prepared_roots)} prepared root(s). '
            f'Files under /kaggle/input: {visible[:40]}'
        )
    ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    if prepared_roots:
        shutil.copytree(prepared_roots[0], ARTIFACT_ROOT, dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(prepared_archives[0]) as archive:
            allowed_roots = {'phase4_b0_artifacts', 'controlled-xnli-10lang-v1'}
            for member in archive.infolist():
                parts = Path(member.filename).parts
                if not parts or parts[0] not in allowed_roots or '..' in parts:
                    raise ValueError('Unexpected file in the prepared-data archive.')
            archive.extractall(WORK_ROOT)
        extracted_root = WORK_ROOT / 'controlled-xnli-10lang-v1'
        if extracted_root.is_dir():
            if ARTIFACT_ROOT.exists() and any(ARTIFACT_ROOT.iterdir()):
                raise FileExistsError(f'Artifact root is not empty: {ARTIFACT_ROOT}')
            if ARTIFACT_ROOT.exists():
                ARTIFACT_ROOT.rmdir()
            extracted_root.replace(ARTIFACT_ROOT)

print('Artifact root:', ARTIFACT_ROOT)

In [ ]:
import json

from phase2.protocol import build_manifest, load_protocol
from phase2.results import validate_result_record
from phase2.run_baseline import validate_prepared_artifacts

protocol = load_protocol(PROTOCOL_PATH)
prepared = validate_prepared_artifacts(protocol, ARTIFACT_ROOT)
run_manifest = build_manifest(protocol)
run_manifest['data_artifacts'] = prepared['data_artifacts']
run_manifest['data_manifest_sha256'] = prepared['data_manifest_sha256']

order_ids = [order['id'] for order in protocol['orders']]
if order_ids != [f'order-{index:03d}' for index in range(1, 6)]:
    raise ValueError('The Phase 4 protocol must contain orders 001-005.')

print({
    'protocol_id': protocol['protocol_id'],
    'protocol_sha256': run_manifest['protocol_sha256'],
    'data_manifest_sha256': prepared['data_manifest_sha256'],
    'orders': order_ids,
    'training_seed': protocol['training']['seeds'],
})

In [ ]:
import subprocess
import sys

results_dir = ARTIFACT_ROOT / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

expected = {
    run['run_id']: run
    for run in run_manifest['runs']
    if run['method_id'] == 'B0'
}
completed_order_ids = set()
for run_id, run in expected.items():
    record_path = results_dir / f'{run_id}.json'
    if record_path.is_file():
        record = json.loads(record_path.read_text(encoding='utf-8'))
        validate_result_record(record, run_manifest)
        completed_order_ids.add(run['order_id'])

next_order_id = next(
    (order_id for order_id in order_ids if order_id not in completed_order_ids),
    None,
)
if next_order_id is not None:
    command = [
        sys.executable, '-m', 'phase2.run_baseline',
        '--method', 'B0',
        '--protocol', str(PROTOCOL_PATH),
        '--artifact-root', str(ARTIFACT_ROOT),
        '--order-id', next_order_id,
    ]
    print(f'Running {next_order_id} only; completed orders: {len(completed_order_ids)}/{len(order_ids)}.')
    subprocess.run(command, cwd=SOURCE_ROOT, check=True)

    run = next(run for run in expected.values() if run['order_id'] == next_order_id)
    record_path = results_dir / f"{run['run_id']}.json"
    if not record_path.is_file():
        raise FileNotFoundError(f'Missing B0 result: {record_path.name}')
    record = json.loads(record_path.read_text(encoding='utf-8'))
    validate_result_record(record, run_manifest)
    completed_order_ids.add(next_order_id)
else:
    print('All Phase 4 B0 orders already have validated results.')

print(f'Validated {len(completed_order_ids)}/{len(order_ids)} Phase 4 B0 order results.')

In [ ]:
progress_path = WORK_ROOT / 'phase4_b0_progress.zip'
with zipfile.ZipFile(progress_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in ARTIFACT_ROOT.rglob('*'):
        if not path.is_file():
            continue
        relative = path.relative_to(ARTIFACT_ROOT)
        if relative.parts and relative.parts[0] in {'protocol_manifest.json', 'data', 'results'}:
            archive.write(path, Path('phase4_b0_artifacts') / relative)
print('Download:', progress_path)